# KPI_Design

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [31]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

order_items_date_columns = ['shipping_limit_date']

orders_master = pd.read_csv('../data/processed/orders_master.csv', parse_dates=date_cols)
items_master = pd.read_csv('../data/processed/items_master.csv', parse_dates=order_items_date_columns)
customers_cleaned   = pd.read_csv('../data/interim/olist_customers_cleaned.csv')


print(orders_master.shape)
print(items_master.shape)

(99441, 23)
(112650, 19)


## Sales KPIs

In [5]:
# Total Orders 
total_orders = orders_master['order_id'].nunique()
total_orders

99441

In [6]:
# Total Items Sold 
total_items_sold = items_master.shape[0]
total_items_sold

112650

In [7]:
# Total Sales
total_sales = orders_master['total_item_price'].sum()
total_sales

np.float64(13591643.7)

In [ ]:
# AOV (Avrage Order Vales)
aov = orders_master['total_item_price'].mean()
aov_with_freight = (orders_master['total_item_price'] + orders_master['total_freight_value']).mean()

print(f"KPI 4 — Average Order Value (items only): R$ {aov:.2f}")
print(f"        Average Order Value (incl. freight): R$ {aov_with_freight:.2f}")

KPI 4 — Average Order Value (items only): R$ 137.75
        Average Order Value (incl. freight): R$ 160.58


## Customer KPIs

In [35]:
# Total Cosumers
total_customers = customers_cleaned['customer_unique_id'].nunique()
total_customers

96096

In [62]:
# Customer Repeat Rate
grouped = customers_cleaned.groupby(customers_cleaned['customer_unique_id'])['customer_id'].size() 
repeat_customers = (grouped > 1).sum()

repeat_customers_rate = repeat_customers / total_customers
print(f"Customer Repeat Rate: {repeat_customers_rate:.2f}%")

Customer Repeat Rate: 0.03%


## Delivery KPIs

In [ ]:
# On-Time Delivery Rate
delivered = orders_master[orders_master['order_delivered_customer_date'].notna()]

on_time_rate = delivered['is_on_time'].mean() * 100

print(f"KPI 1 — On-Time Delivery Rate: {on_time_rate:.2f}%")
print(f"Base population: {len(delivered)} delivered orders (out of {len(orders_master)} total)")

KPI 1 — On-Time Delivery Rate: 91.89%
Base population: 96476 delivered orders (out of 99441 total)


In [ ]:
# Avrage Delivery Time 
avg_delivery_hours = delivered['purchase_to_customer_hours'].mean()
median_delivery_hours = delivered['purchase_to_customer_hours'].median()

print(f"KPI 2 — Avg. Delivery Time: {avg_delivery_hours/24:.1f} days (mean)")
print(f"                             {median_delivery_hours/24:.1f} days (median)")

KPI 2 — Avg. Delivery Time: 12.6 days (mean)
                             10.2 days (median)


In [76]:
is_late_rate = delivered['is_late'].mean() * 100

print(f"is_late_rate Delivery Rate: {is_late_rate:.2f}%")
print(f"Base population: {len(delivered)} delivered orders (out of {len(orders_master)} total)")

is_late_rate Delivery Rate: 6.77%
Base population: 96476 delivered orders (out of 99441 total)


## Customer Satisfaction KPIs

In [27]:
# Average Review Score
avg_review_score = orders_master['review_score'].mean()
n_reviews = orders_master['review_score'].notna().sum()

print(f"KPI 3 — Avg. Review Score: {avg_review_score:.2f} / 5")
print(f"Based on {n_reviews} orders with a review ({n_reviews/len(orders_master)*100:.1f}% of all orders)")

KPI 3 — Avg. Review Score: 4.09 / 5
Based on 98673 orders with a review (99.2% of all orders)


In [78]:
low_rating_rate = (
    orders_master["review_score"].le(2).mean() * 100
)
low_rating_rate
print(f"1-2 star Review Rate: {low_rating_rate:.2f}%")

1-2 star Review Rate: 14.58%


## Seller KPIs

In [ ]:
items_master.sample()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_photos_qty,product_description_lenght,volume_cm3,freight_ratio,distance_km,is_late,review_score
91480,cf8ceeefc490504b4e117676aabb2820,1,8562e2c780a345b609e1b802d7e3e4ae,8160255418d5aaa7dbdc9f4c64ebda44,2018-01-26 13:47:19,79.9,16.32,bed_bath_table,1100.0,45.0,8.0,35.0,1.0,1208.0,12600.0,0.204255,531.742736,False,5.0


In [81]:
# Total Seller
total_seller = items_master['seller_id'].nunique()
total_seller

3095

In [86]:
grouped = items_master.groupby(items_master['seller_id'])['order_id'].size()
grouped.mean()

np.float64(36.39741518578352)

In [30]:
kpis = {
    "Total Orders": total_orders,
    "Total Sales": total_sales,
    "Average Order Value": aov,
    "On-Time Delivery Rate": on_time_rate,
    "Average Review Score": avg_review_score,
    "Low Rating Rate": low_rating_rate
}

kpis

kpi_df = pd.DataFrame(
    kpis.items(),
    columns=["KPI", "Value"]
)

kpi_df

,KPI,Value
0,Total Orders,9.944100e+04
1,Total Sales,1.359164e+07
2,Average Order Value,1.377541e+02
3,On-Time Delivery Rate,9.188710e+01
4,Average Review Score,4.086265e+00
5,Low Rating Rate,1.457648e+01
